# Day 12: Semantic Search and Vector Databases

Welcome to Day 12! Today, we transition from foundational LLM usage to a core pillar of modern AI Engineering: **Semantic Search**. This is the mechanism that powers Retrieval-Augmented Generation (RAG) and allows applications to "understand" the meaning of queries rather than just matching keywords.


## Core Theory (Just-in-Time)

### What is Semantic Search?
Traditional search engines (like Elasticsearch using BM25) rely on **keyword matching**. If you search for "dog", it looks for the exact word "dog". 

**Semantic search**, on the other hand, understands *intent and context*. If you search for "canine companion", a semantic search engine knows this is related to "dog" even if the words don't match.

### How does it work?
1. **Embeddings:** Text is converted into dense vector representations (arrays of floating-point numbers) using an embedding model. These vectors represent the semantic meaning of the text.
2. **Vector Space:** Similar concepts are placed close together in a high-dimensional vector space.
3. **Similarity Search:** When a user queries the system, the query is also converted into a vector. The system then calculates the distance (e.g., Cosine Similarity, Euclidean Distance) between the query vector and all document vectors in the database. The closest vectors are returned as the most relevant results.

### Enter Vector Databases (Qdrant)
To store and search these high-dimensional vectors efficiently at scale, we use specialized **Vector Databases**. In our stack, we use **Qdrant**, a high-performance vector search engine written in Rust.


## Code Implementation

We will implement a complete, production-grade semantic search script using Python, LangChain, and Qdrant. For this example, we will use local memory for Qdrant and HuggingFace's open-source embeddings, so you can run it completely locally without external API keys.

Let's set up our imports and type hints.


In [1]:
import uuid
from typing import List

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# Initialize local in-memory Qdrant client
# In production, you would connect to a dedicated Qdrant server or Qdrant Cloud.
qdrant_client = QdrantClient(":memory:")

# Initialize our embedding model.
# We use a lightweight open-source model from HuggingFace for local execution.
embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

### Preparing the Data

In a production scenario, you would extract text from PDFs, websites, or databases. For today, we will use a local batch of predefined text documents.


In [2]:
def prepare_documents() -> List[Document]:
    """
    Prepares a batch of local text documents for vectorization.
    
    Returns:
        List[Document]: A list of LangChain Document objects containing page_content and metadata.
    """
    raw_texts = [
        "The quick brown fox jumps over the lazy dog.",
        "A fast, dark-colored canine leaps across a sleeping hound.",
        "Artificial Intelligence is transforming software engineering rapidly.",
        "Python is a versatile programming language widely used in data science.",
        "Qdrant is a high-performance open-source vector search engine."
    ]
    
    # Wrap strings into LangChain Document objects with basic metadata
    docs = []
    for i, text in enumerate(raw_texts):
        doc = Document(
            page_content=text,
            metadata={"source_id": i, "category": "general"}
        )
        docs.append(doc)
        
    return docs

documents = prepare_documents()
print(f"Prepared {len(documents)} documents.")


Prepared 5 documents.


### Creating the Vector Store and Inserting Documents

Now we will create a Qdrant collection and ingest our documents. The `QdrantVectorStore` wrapper from LangChain handles the conversion of text to vectors (using our `embedding_model`) and the insertion process automatically.


In [3]:
def build_vector_store(
    docs: List[Document], 
    client: QdrantClient, 
    embeddings: HuggingFaceEmbeddings
) -> QdrantVectorStore:
    """
    Builds and populates a Qdrant Vector Store from a list of documents.
    
    Args:
        docs (List[Document]): The documents to ingest.
        client (QdrantClient): The Qdrant client instance.
        embeddings (HuggingFaceEmbeddings): The embedding model to use.
        
    Returns:
        QdrantVectorStore: The initialized and populated vector store.
    """
    collection_name = "day_12_collection"
    
    # Create the collection explicitly to define vector dimensions and distance metric.
    # The "all-MiniLM-L6-v2" model produces vectors of 384 dimensions.
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    )
    
    # Initialize the LangChain Qdrant wrapper
    vector_store = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=embeddings,
    )
    
    # Add documents to the store
    # We generate UUIDs to ensure each document has a unique identifier in the database.
    uuids = [str(uuid.uuid4()) for _ in range(len(docs))]
    vector_store.add_documents(documents=docs, ids=uuids)
    
    return vector_store

vector_store = build_vector_store(documents, qdrant_client, embedding_model)
print("Successfully built vector store and ingested documents.")


Successfully built vector store and ingested documents.


### Performing Semantic Search

With our database populated, we can now run a query. Notice how we search for "A sleeping puppy" and it correctly identifies the document about the "lazy dog" or "sleeping hound", despite not sharing the exact keywords.


In [4]:
def perform_search(vector_store: QdrantVectorStore, query: str, k: int = 2) -> None:
    """
    Executes a semantic search against the vector store and prints the top results.
    
    Args:
        vector_store (QdrantVectorStore): The vector store to search.
        query (str): The search query text.
        k (int, optional): The number of top results to return. Defaults to 2.
    """
    print(f"\n--- Searching for: '{query}' ---")
    
    # Perform similarity search with score
    # Returns a list of tuples containing (Document, similarity_score)
    results = vector_store.similarity_search_with_score(query, k=k)
    
    for i, (doc, score) in enumerate(results, 1):
        # Qdrant uses Cosine distance internally, but LangChain often standardizes it.
        # Higher score typically means higher similarity.
        print(f"\nResult {i} (Score: {score:.4f}):")
        print(f"Content: {doc.page_content}")
        print(f"Metadata: {doc.metadata}")

# Test 1: Semantic understanding
perform_search(vector_store, "A sleeping puppy")

# Test 2: Domain specific knowledge
perform_search(vector_store, "Tools for AI programming")



--- Searching for: 'A sleeping puppy' ---

Result 1 (Score: 0.5735):
Content: A fast, dark-colored canine leaps across a sleeping hound.
Metadata: {'source_id': 1, 'category': 'general', '_id': 'b9099d9b-7698-41b3-bb77-00e036a9168e', '_collection_name': 'day_12_collection'}

Result 2 (Score: 0.2616):
Content: The quick brown fox jumps over the lazy dog.
Metadata: {'source_id': 0, 'category': 'general', '_id': 'e1da95f7-c52a-4e51-b3cf-3c95642caa37', '_collection_name': 'day_12_collection'}

--- Searching for: 'Tools for AI programming' ---

Result 1 (Score: 0.5063):
Content: Artificial Intelligence is transforming software engineering rapidly.
Metadata: {'source_id': 2, 'category': 'general', '_id': 'f5989d07-bad5-42c8-91ca-99ae3981fbdd', '_collection_name': 'day_12_collection'}

Result 2 (Score: 0.4022):
Content: Python is a versatile programming language widely used in data science.
Metadata: {'source_id': 3, 'category': 'general', '_id': 'b6b10ffd-b383-4e25-a12c-92d21cf45f8e', '_col

## Common Pitfalls (Production Considerations)

1. **Embedding Dimension Mismatches:** You must ensure the vector size defined in your Qdrant collection exactly matches the output dimension of your chosen embedding model. If you switch models (e.g., from `all-MiniLM-L6-v2` [384] to OpenAI `text-embedding-3-small` [1536]), you must recreate the collection or use different namespaces.
2. **Chunking Strategy Ignored:** In this example, our documents were already single sentences. In production, feeding massive documents directly into an embedding model will truncate them (due to token limits) and dilute the semantic meaning. You *must* split large text into smaller chunks before vectorizing.
3. **Keyword Blindness:** Pure semantic search struggles with exact keyword lookups (e.g., searching for a specific UUID, an acronym, or an exact part number). Production systems often use "Hybrid Search" (combining semantic and keyword search).
4. **Forgetting Metadata:** Without metadata, you cannot filter searches (e.g., "Search only documents authored by John in 2023"). Always enrich your chunks with structured metadata before insertion.


## Practical Lab / Homework

Your task for today is to build a robust class-based semantic search utility.

**Instructions:**
1. Create a class `LocalSemanticSearcher`.
2. The `__init__` method should take a list of raw strings, initialize an in-memory Qdrant client, load the `HuggingFaceEmbeddings` model, and populate the vector store.
3. Implement a `search` method that takes a `query` string and an integer `top_n`, returning the matching `Document` objects and their scores.
4. Add a feature to filter by metadata: modify your setup to include a `topic` metadata field for each string, and implement a `search_by_topic` method that only searches within a specific topic. *(Hint: Look into Qdrant's filtering capabilities or LangChain's `filter` kwarg in `similarity_search`)*.
5. Provide strict type hinting and docstrings for all methods. Do not mock any methods. Write a simple executable block at the bottom to prove it works.

*Write your implementation in the code cell below:*


In [5]:
# LAB IMPLEMENTATION
from typing import List, Tuple, Dict, Any
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, Filter, FieldCondition, MatchValue
import uuid

class LocalSemanticSearcher:
    """
    A robust, class-based utility for semantic search using Qdrant and LangChain.
    """
    
    def __init__(self, texts_with_metadata: List[Dict[str, Any]], collection_name: str = "lab_collection"):
        """
        Initializes the semantic searcher, embedding model, and populates the vector store.
        
        Args:
            texts_with_metadata (List[Dict[str, Any]]): A list of dictionaries containing 'text' and 'metadata'.
            collection_name (str, optional): The name of the Qdrant collection. Defaults to "lab_collection".
        """
        self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
        self.client = QdrantClient(":memory:")
        self.collection_name = collection_name
        
        self.client.create_collection(
            collection_name=self.collection_name,
            vectors_config=VectorParams(size=384, distance=Distance.COSINE),
        )
        
        self.vector_store = QdrantVectorStore(
            client=self.client,
            collection_name=self.collection_name,
            embedding=self.embeddings,
        )
        
        # Prepare and insert documents
        docs = []
        for item in texts_with_metadata:
            doc = Document(
                page_content=item["text"],
                metadata=item.get("metadata", {})
            )
            docs.append(doc)
            
        uuids = [str(uuid.uuid4()) for _ in range(len(docs))]
        self.vector_store.add_documents(documents=docs, ids=uuids)

    def search(self, query: str, top_n: int = 3) -> List[Tuple[Document, float]]:
        """
        Performs a semantic search for the given query.
        
        Args:
            query (str): The search query.
            top_n (int, optional): The number of results to return. Defaults to 3.
            
        Returns:
            List[Tuple[Document, float]]: A list of tuples containing the matched Document and its score.
        """
        return self.vector_store.similarity_search_with_score(query, k=top_n)
        
    def search_by_topic(self, query: str, topic: str, top_n: int = 3) -> List[Tuple[Document, float]]:
        """
        Performs a semantic search restricted to a specific topic using metadata filtering.
        
        Args:
            query (str): The search query.
            topic (str): The exact topic to filter by.
            top_n (int, optional): The number of results to return. Defaults to 3.
            
        Returns:
            List[Tuple[Document, float]]: A list of tuples containing the matched Document and its score.
        """
        # LangChain's QdrantVectorStore supports passing Qdrant filter objects directly
        qdrant_filter = Filter(
            must=[
                FieldCondition(
                    key="metadata.topic",
                    match=MatchValue(value=topic),
                )
            ]
        )
        
        return self.vector_store.similarity_search_with_score(
            query, 
            k=top_n, 
            filter=qdrant_filter
        )

# Execution block to prove it works
if __name__ == "__main__":
    dataset = [
        {"text": "The mitochondria is the powerhouse of the cell.", "metadata": {"topic": "biology"}},
        {"text": "Photosynthesis converts light energy into chemical energy.", "metadata": {"topic": "biology"}},
        {"text": "Newton's second law is F = ma.", "metadata": {"topic": "physics"}},
        {"text": "Quantum mechanics describes nature at the smallest scales.", "metadata": {"topic": "physics"}}
    ]
    
    searcher = LocalSemanticSearcher(dataset)
    
    print("--- General Search ---")
    results = searcher.search("energy conversion", top_n=2)
    for doc, score in results:
        print(f"Score: {score:.4f} | {doc.page_content}")
        
    print("\n--- Filtered Search (Topic: physics) ---")
    filtered_results = searcher.search_by_topic("force and mass", topic="physics", top_n=1)
    for doc, score in filtered_results:
        print(f"Score: {score:.4f} | {doc.page_content}")



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

--- General Search ---
Score: 0.5385 | Photosynthesis converts light energy into chemical energy.
Score: 0.2349 | The mitochondria is the powerhouse of the cell.

--- Filtered Search (Topic: physics) ---
Score: 0.6238 | Newton's second law is F = ma.
